# Retrieval, Tool Use, and Agentic Systems

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/parisar/BME6938-Advanced-Biomedical-AI-Spring-2027/blob/main/lectures/notebooks/week09_retrieval_agents.ipynb)

*Click the badge to open this notebook in Google Colab, then choose File > Save a copy in Drive so your changes are kept.*

**BME 6938 Advanced Biomedical AI · Week 9 · Reference notebook (ungraded)**

This notebook follows the Week 9 lecture and its three clips. It builds retrieval-augmented generation on a small set of
synthetic hospital protocols, runs a ReAct agent with two read-only tools on synthetic patients, and measures how errors
compound over steps.

| Section | Clip | What you will see |
|---|---|---|
| 1 | Retrieval-augmented generation | Chunks, embeddings, cosine similarity, a threshold, and answers with and without the retrieved source |
| 2 | Reason, act, observe | A JSON tool-call parser, a ReAct loop driven by a small open model (SmolLM2-1.7B-Instruct), and how often its calls parse and its answers are right |
| 3 | Errors compound over steps | A simulation of 0.95ⁿ and 0.99ⁿ, and three mitigations: a repeat guard, read-only tools, and a clinician checkpoint |

**How to run it**
- **Google Colab or HiPerGator:** *Runtime > Run all*. End to end it took about 14 minutes of CPU time when we ran it, on a
  20-thread desktop with both models already downloaded; the two agent runs in sections 2.5 and 3.2 are about 11 of those
  minutes. Expect longer on a shared machine: Colab's free CPU runtime gives you about two cores, so treat the 14 minutes as
  a floor rather than an estimate for your own session. The first run downloads two open models from Hugging Face: an
  embedding model (about 90 MB) and SmolLM2-1.7B-Instruct (about 3.4 GB, which takes about 6.4 GB of memory once it is
  loaded in float32).
- **No model download?** Set `USE_LLM = False` in section 0. The agent then runs as a scripted policy, so the notebook
  still runs end to end; only the language-model answers are skipped.
- **Data.** The protocols, patients, lab values, and medications are invented: no real patient data.

**Credits.** The thought, action, observation format follows Yao et al., ReAct (ICLR 2023). The chunking, embedding, and
threshold design follows Zakka et al., Almanac (NEJM AI 2024). The code is written for this course.

## 0. Setup

In [ ]:
import json, math, re, time                                  # standard library
import numpy as np                                           # arrays
import torch                                                 # tensors
import torch.nn.functional as F                              # normalization of embeddings
import matplotlib.pyplot as plt                              # plots
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM   # open models from Hugging Face

USE_LLM = True                                               # False: skip SmolLM2-1.7B and run the scripted agent instead
torch.manual_seed(0)                                         # same results on every run
rng = np.random.default_rng(0)                               # random numbers for section 3
T_START = time.time()                                        # to report the total run time at the end
print("PyTorch", torch.__version__)

## 1. Retrieval-augmented generation  *(clip 1)*

### 1.1 The hospital's documents, split into chunks
A model trained on public text cannot know this hospital's own rules. Our synthetic hospital has six short protocols, already
split into 12 chunks, one per section. Almanac split web articles into chunks of 1,000 tokens; ours are one or two sentences
so that you can read them all. Every chunk keeps its source, so that an answer can cite it.

In [ ]:
CHUNKS = [                                                   # (source, text): an invented protocol collection
    ("Sepsis protocol, section 1", "Screen every adult in the emergency department for sepsis with the qSOFA score and a lactate level."),
    ("Sepsis protocol, section 2", "Start broad-spectrum antibiotics within 1 hour of recognizing sepsis or septic shock, after drawing blood cultures."),
    ("Sepsis protocol, section 3", "Give 30 mL/kg of intravenous crystalloid fluid within the first 3 hours for hypotension or a lactate of 4 mmol/L or more."),
    ("Sepsis protocol, section 4", "Start norepinephrine if the mean arterial pressure stays below 65 mmHg after fluids."),
    ("Potassium protocol, section 1", "A potassium above 5.0 mmol/L is high. Repeat the test to rule out a hemolyzed sample and obtain an ECG."),
    ("Potassium protocol, section 2", "For potassium of 6.0 mmol/L or more, or ECG changes, give intravenous calcium gluconate and notify the physician."),
    ("Insulin protocol, section 1", "Check capillary glucose before meals and at bedtime for patients on basal-bolus insulin."),
    ("Insulin protocol, section 2", "Treat glucose below 70 mg/dL with 15 g of fast-acting carbohydrate and recheck in 15 minutes."),
    ("Falls protocol, section 1", "Score every inpatient with the Morse Fall Scale on admission and after any change in condition."),
    ("Falls protocol, section 2", "Patients with a Morse score of 45 or more get a bed alarm and non-slip socks."),
    ("VTE protocol, section 1", "Give enoxaparin 40 mg daily for VTE prophylaxis unless the patient is bleeding or has a platelet count below 50,000."),
    ("Pressure injury protocol, section 1", "Turn bed-bound patients every 2 hours and check the skin over the sacrum and heels each shift."),
]
for src, text in CHUNKS[:3]:                                 # show the first three chunks
    print(f"[{src}] {text}")
print("...", len(CHUNKS), "chunks in total")

### 1.2 Embed each chunk once
An embedding model turns each chunk into a vector, so that texts with similar meaning get similar vectors. We use the small open
model `all-MiniLM-L6-v2` (384 numbers per text); Almanac used OpenAI's `text-embedding-ada-002` (1,536 numbers). The vector of a
text is the average of its token vectors, scaled to length 1. The chunks are embedded **once** and stored; only the question is
embedded at question time.

In [ ]:
EMB_NAME = "sentence-transformers/all-MiniLM-L6-v2"          # small open embedding model (about 90 MB)
emb_tok = AutoTokenizer.from_pretrained(EMB_NAME)            # its tokenizer
emb_model = AutoModel.from_pretrained(EMB_NAME).eval()       # its encoder, in evaluation mode

@torch.no_grad()                                             # no gradients needed: we only run the model
def embed(texts):
    """Mean of the token vectors, scaled to length 1: one row per text."""
    batch = emb_tok(texts, padding=True, truncation=True, return_tensors="pt")   # token ids and the padding mask
    h = emb_model(**batch).last_hidden_state                 # (texts, tokens, 384) token vectors
    mask = batch["attention_mask"].unsqueeze(-1).float()     # 1 for real tokens, 0 for padding
    v = (h * mask).sum(1) / mask.sum(1)                      # average over the real tokens only
    return F.normalize(v, dim=-1).numpy()                    # length 1, so a dot product is the cosine similarity

E = embed([text for _, text in CHUNKS])                      # all chunks, embedded once
print("chunk embeddings:", E.shape, "(chunks, numbers per chunk)")
print("first 6 numbers of chunk 2:", E[1, :6].round(3))

### 1.3 Embed the question and score every chunk by cosine similarity
Cosine similarity is the dot product of two vectors divided by both lengths. Our vectors already have length 1, so the scores of
all chunks are one matrix-vector product. The clip's scores (0.86, 0.79, 0.62) were illustrative; here are the real ones for this
model.

In [ ]:
def cosine(a, b):
    """Cosine similarity from its definition: a · b / (|a| |b|)."""
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

def retrieve(question, k=3):
    """The k best chunks for a question: list of (score, chunk index)."""
    q = embed([question])[0]                                 # the question's vector
    scores = E @ q                                           # cosine similarity with every chunk at once
    best = np.argsort(-scores)[:k]                           # indices of the k highest scores
    return [(float(scores[i]), int(i)) for i in best]

QUESTION = "Within how many hours must antibiotics start in our sepsis protocol?"   # the clip's question
top = retrieve(QUESTION)                                     # top 3 chunks
for s, i in top:
    print(f"{s:.2f}  [{CHUNKS[i][0]}] {CHUNKS[i][1][:60]}...")
q = embed([QUESTION])[0]                                     # check one score with the formula
print("\ncosine from the definition for the best chunk:", round(cosine(q, E[top[0][1]]), 2))

### 1.4 Only chunks above a threshold pass
Almanac passed only chunks scoring above **λ = 0.83**. That value belongs to its embedding model: every model has its own scale of
scores, so a threshold must be chosen again for a new model. We choose ours from eight questions, five that the protocols answer
and three that they do not, and put the threshold halfway between the two groups.

In [ ]:
CALIBRATION = [                                              # (question, do the protocols answer it?)
    ("Within how many hours must antibiotics start in our sepsis protocol?", True),
    ("How much IV fluid do we give for a lactate of 4?", True),
    ("What do we do for a potassium of 5.6?", True),
    ("When do we start norepinephrine?", True),
    ("How do we treat a low blood sugar?", True),
    ("Do we have a protocol for rattlesnake bites?", False),
    ("What is our visiting policy for the ICU?", False),
    ("How do we manage a pediatric asthma attack?", False),
]
best = [(retrieve(qq, k=1)[0][0], ok) for qq, ok in CALIBRATION]   # best score of each question
for (qq, ok), (s, _) in zip(CALIBRATION, best):
    print(f"{s:.2f}  {'answerable  ' if ok else 'unanswerable'}  {qq}")
low_yes = min(s for s, ok in best if ok)                     # weakest answerable question
high_no = max(s for s, ok in best if not ok)                 # strongest unanswerable question
LAMBDA = round((low_yes + high_no) / 2, 2)                   # halfway between the two groups
print(f"\nanswerable from {low_yes:.2f}, unanswerable up to {high_no:.2f}: threshold λ = {LAMBDA}")

### 1.5 The model alone, then with the retrieved source
Now a language model answers. We use **SmolLM2-1.7B-Instruct**, a small open model that runs on a CPU. First it answers alone, then with the
passing chunk pasted into the prompt together with its source. When no chunk passes, there are two designs, as in the clip: the
published Almanac falls back on the model's own knowledge, and the preprint version abstained.

In [ ]:
LM_NAME = "HuggingFaceTB/SmolLM2-1.7B-Instruct"              # small open instruction-tuned model (about 3.4 GB)
if USE_LLM:
    lm_tok = AutoTokenizer.from_pretrained(LM_NAME)          # its tokenizer and chat template
    lm = AutoModelForCausalLM.from_pretrained(LM_NAME, dtype=torch.float32).eval()   # the model, on the CPU

@torch.no_grad()                                             # no gradients: generation only
def chat(messages, max_new_tokens=90):
    """One greedy reply from the language model to a list of chat messages."""
    ids = lm_tok.apply_chat_template(messages, add_generation_prompt=True,
                                     return_tensors="pt", return_dict=True)["input_ids"]
    out = lm.generate(ids, max_new_tokens=max_new_tokens, do_sample=False)   # greedy: same answer on every run
    return lm_tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip()

def rag_answer(question, mode="fallback"):
    """Retrieve, apply the threshold, and answer; mode is 'fallback' (published Almanac) or 'abstain' (preprint)."""
    passing = [(s, i) for s, i in retrieve(question) if s >= LAMBDA]   # chunks above the threshold
    if not passing:
        if mode == "abstain":
            return "Not found in the sources.", []           # the preprint's behavior
        return chat([{"role": "user", "content": question}]), []   # the published behavior: the model alone
    sources = "\n".join(f"[{CHUNKS[i][0]}] {CHUNKS[i][1]}" for _, i in passing)   # passing chunks with their sources
    prompt = (f"Answer the question using only these sources, and cite the source in brackets.\n\n{sources}\n\n"
              f"Question: {question}")
    return chat([{"role": "user", "content": prompt}]), [CHUNKS[i][0] for _, i in passing]

if USE_LLM:
    print("MODEL ALONE:\n", chat([{"role": "user", "content": QUESTION}]), "\n")
    answer, used = rag_answer(QUESTION)
    print("WITH RETRIEVAL (sources:", used, "):\n", answer, "\n")
    snake = "Do we have a protocol for rattlesnake bites?"   # the clip's second question: nothing passes
    print("NOTHING PASSES, abstain: ", rag_answer(snake, mode="abstain")[0])
    print("NOTHING PASSES, fallback:", rag_answer(snake, mode="fallback")[0][:300])

Read the answers. When we ran this, the model alone hedged: antibiotics "within 1-2 hours of diagnosis", a textbook range and
not this hospital's rule (the clip's "3 hours" was illustrative). With the passing chunks in the prompt it answered "within 1
hour of recognizing sepsis or septic shock, after drawing blood cultures", which is section 2 word for word. Notice what it did
not do: the prompt asks for the source in brackets, and the model named the protocol in prose instead. Asking for a citation
does not guarantee one in the form you asked for, which is why the program, not the model, keeps the list of sources it used.
Two chunks passed here (0.78 and 0.46), because our threshold is lower than Almanac's. For the rattlesnake question nothing
passes: the abstaining design says so, and the fallback design answers from the model's own knowledge. Read that fallback
answer carefully. It is fluent, it is general first aid rather than anything from this hospital, and there is no source beside
it to check it against. That absence is the whole difference between the two designs.

## 2. Reason, act, observe  *(clip 2)*

### 2.1 Two read-only tools on synthetic patients
The agent can call two tools: `get_labs` returns the last results of a lab test, and `get_meds` returns the current
medications. Both only **read** the record. P-001 is the clip's patient.

In [ ]:
LABS = {                                                     # invented results, oldest first
    "P-001": {"potassium": [4.6, 5.1, 5.6], "creatinine": [1.1, 1.3, 1.6], "sodium": [138, 137, 139]},   # mmol/L, mg/dL, mmol/L
    "P-002": {"potassium": [4.1, 4.0, 3.9], "glucose": [182, 240, 265], "sodium": [131, 128, 126]},      # glucose in mg/dL
    "P-003": {"potassium": [3.6, 3.3, 3.0], "creatinine": [0.9, 0.9, 1.0], "glucose": [95, 101, 98]},
}
MEDS = {"P-001": ["lisinopril 20 mg daily", "spironolactone 25 mg daily"],   # both can raise potassium
        "P-002": ["insulin glargine 20 units nightly"],
        "P-003": ["furosemide 40 mg twice daily"]}           # can lower potassium

def get_labs(patient_id, test, last=3):
    """The last `last` results of a lab test, as text."""
    values = LABS.get(patient_id, {}).get(test)              # None if the patient or test is unknown
    if values is None:
        return f"no {test} results for {patient_id}"
    return f"{test}: " + ", ".join(str(v) for v in values[-int(last):])

def get_meds(patient_id):
    """The current medications, as text."""
    return "; ".join(MEDS.get(patient_id, ["no medications found"]))

TOOLS = {"get_labs": get_labs, "get_meds": get_meds}         # the only tools the program will run
print(get_labs("P-001", "potassium", 3))
print(get_meds("P-001"))

### 2.2 A tool call is text: the parser
The model writes a line `Action: {...}` with a JSON object, exactly as in the clip. The **program** finds that line, parses the
JSON, and checks that the tool exists before running anything. Anything else is rejected with a reason, which the agent sees as its
observation.

In [ ]:
def parse_action(text):
    """Find 'Action: {...}' in the model's reply; return (action dict, 'ok') or (None, reason)."""
    m = re.search(r"Action:\s*(\{.*\})", text, re.S)         # the JSON object after 'Action:'
    if not m:
        return None, "no Action line"
    try:
        action = json.loads(m.group(1))                      # strict JSON parsing
    except json.JSONDecodeError as e:
        return None, f"invalid JSON ({e.msg})"
    if not isinstance(action, dict) or not isinstance(action.get("args"), dict):
        return None, "missing args"
    if action.get("tool") not in list(TOOLS) + ["finish"]:   # only known tools, plus finish
        return None, f"unknown tool {action.get('tool')!r}"
    return action, "ok"

tests = ['Thought: I need the labs.\nAction: {"tool": "get_labs", "args": {"patient_id": "P-001", "test": "potassium", "last": 3}}',
         "Action: {'tool': 'get_labs'}",                     # single quotes are not valid JSON
         'Action: {"tool": "place_order", "args": {"order": "potassium chloride"}}',   # not a tool the agent has
         "I think the potassium is fine."]                   # no action at all
for t in tests:
    print(parse_action(t)[1], "<-", t.splitlines()[-1][:70])

### 2.3 The prompt: a few worked examples, as in ReAct
ReAct used no training: the prompt shows the format with a worked example, and the model continues it. Our example uses a
different patient and test (sodium of P-009), so that the model cannot copy the answer.

In [ ]:
SYSTEM = """You are an assistant that answers questions about hospital patients by calling tools.
Tools:
- get_labs(patient_id, test, last): the last results of a lab test
- get_meds(patient_id): the current medications
Each reply must be exactly two lines:
Thought: <one sentence>
Action: <one JSON object>
The JSON is {"tool": "get_labs", "args": {"patient_id": "...", "test": "...", "last": 3}} or {"tool": "get_meds", "args": {"patient_id": "..."}}.
When you know the answer, use {"tool": "finish", "args": {"answer": "..."}}.

Example
Question: Is P-009's sodium falling?
Thought: I need the last three sodium results.
Action: {"tool": "get_labs", "args": {"patient_id": "P-009", "test": "sodium", "last": 3}}
Observation: sodium: 136, 133, 129
Thought: Sodium fell from 136 to 129 mmol/L, below the usual lower limit of 135.
Action: {"tool": "finish", "args": {"answer": "Yes, sodium is falling: 136, 133, 129 mmol/L; 129 is low."}}"""
print(len(SYSTEM.splitlines()), "lines in the system prompt")   # the format, the tools, and one worked example

### 2.4 The ReAct loop, and a scripted fallback
Each step: the model writes a thought and an action; the program parses the action, runs the tool, and appends the result as an
observation; repeat until `finish` or a step limit. The **scripted agent** follows a fixed rule (look up the test named in the
question, then answer from the values) and stands in for the model when `USE_LLM = False`, or for a single step when the model's
reply has failed to parse twice in a row. Every step records who wrote it.

In [ ]:
LIMITS = {"potassium": (3.5, 5.0), "sodium": (135, 145), "glucose": (70, 140), "creatinine": (0.6, 1.2)}   # usual adult ranges

def scripted_step(question, history):
    """A fixed rule: fetch the named test (or the medications), then answer from the values."""
    pid = re.search(r"P-\d{3}", question).group(0)           # the patient named in the question
    test = next((t for t in LIMITS if t in question.lower()), None)   # the lab named in the question, if any
    if not history:                                          # first step: fetch the data
        if test is None:
            return 'Thought: I need the current medications.\nAction: {"tool": "get_meds", "args": {"patient_id": "%s"}}' % pid
        return ('Thought: I need the last three %s results.\nAction: {"tool": "get_labs", "args": {"patient_id": "%s", "test": "%s", "last": 3}}'
                % (test, pid, test))
    obs = history[-1]                                        # the last observation
    if test is None:
        return 'Thought: I have the medications.\nAction: {"tool": "finish", "args": {"answer": "%s"}}' % obs
    v = [float(x) for x in obs.split(":")[1].split(",")]     # the values, oldest first
    lo, hi = LIMITS[test]
    trend = "rising" if v[-1] > v[0] else "falling" if v[-1] < v[0] else "stable"
    state = "above the usual upper limit" if v[-1] > hi else "below the usual lower limit" if v[-1] < lo else "within the usual range"
    return (f"Thought: {test.capitalize()} is {trend}, and {v[-1]:g} is {state}.\n"
            f'Action: {{"tool": "finish", "args": {{"answer": "{test.capitalize()} is {trend}: {obs.split(":")[1].strip()}; {v[-1]:g} is {state}."}}}}')

def react(question, max_steps=4, guard=False, verbose=False):
    """Run the loop; returns (answer or None, list of (author, parse result) per step)."""
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": f"Question: {question}"}]
    observations, log, seen, fails = [], [], set(), 0
    for _ in range(max_steps):
        if USE_LLM and fails < 2:
            reply, author = chat(messages), "model"          # the model writes the next step
        else:
            reply, author, fails = scripted_step(question, observations), "script", 0   # the fallback writes it
        action, why = parse_action(reply)                    # the program reads the action
        fails = fails + 1 if action is None else 0           # count parse failures in a row
        if verbose:
            print(f"[{author}]\n{reply}")
        messages.append({"role": "assistant", "content": reply})
        if action is None:
            obs = f"Error: {why}. Reply with a Thought line and an Action line with valid JSON."
        elif action["tool"] == "finish":
            log.append((author, why))
            return action["args"].get("answer"), log
        else:
            key = json.dumps(action, sort_keys=True)          # the exact call, for the repeat guard in section 3
            if guard and key in seen:
                obs, why = "You already called this tool with these arguments. Use the result you have and finish.", "repeat"
            else:
                seen.add(key)
                try:
                    obs = TOOLS[action["tool"]](**action["args"])   # the program runs the tool
                except TypeError as e:
                    obs, why = f"Error: bad arguments ({e})", "bad arguments"
        log.append((author, why))
        observations.append(obs)
        if verbose:
            print("Observation:", obs)
        messages.append({"role": "user", "content": f"Observation: {obs}"})
    return None, log                                         # step limit reached without finish

answer, log = react("Is P-001's potassium rising, and is it abnormal?", verbose=True)   # the clip's task
print("\nANSWER:", answer)

Compare this run with the clip. The format is right and the first tool call parses, and then read what the model wrote after it:
an `Observation:` line of its own, with potassium values (6.5, 6.8, 7.1) that no tool returned. The program ignores that line and
appends the real observation, 4.6, 5.1, 5.6. From there the model repeats the same call until the step limit and never finishes,
so this task, the clip's own task, is the one task of the eight it does not answer. A well-formed tool call says nothing about
whether what surrounds it is reasoning.

### 2.5 Eight tasks: does the call parse, does the agent finish, is the answer right?
We run eight questions about three patients and check three things separately. In this record the true answer to every lab
question is yes, so an answer counts as right if it does not start with "No" and either starts with "Yes" or names the finding
(for example "above" or "rising"). A medication answer counts as right if it names every drug. These checks are deliberately
simple: they do not catch a wrong unit, for example.

In [ ]:
def says(answer, *words):
    """A yes answer: not starting with 'no', and starting with 'yes' or naming one of the words."""
    a = answer.lower()
    return not a.startswith("no") and (a.startswith("yes") or any(w in a for w in words))

TASKS = [  # (question, check of the answer); the true answer to every lab question is yes
    ("Is P-001's potassium rising, and is it abnormal?", lambda a: says(a, "above", "high")),
    ("Is P-001's creatinine rising?",                   lambda a: says(a, "rising")),
    ("What medications is P-001 taking?",               lambda a: "lisinopril" in a.lower() and "spironolactone" in a.lower()),
    ("Is P-002's sodium falling?",                      lambda a: says(a, "falling")),
    ("Is P-002's glucose above 200 mg/dL?",             lambda a: says(a, "above")),
    ("What medications is P-002 taking?",               lambda a: "glargine" in a.lower()),
    ("Is P-003's potassium low?",                       lambda a: says(a, "below", "low")),
    ("What medications is P-003 taking?",               lambda a: "furosemide" in a.lower()),
]

def evaluate(guard=False):
    """Run all tasks; print one row per task and return the totals."""
    steps = parsed = finished = correct = 0
    for question, check in TASKS:
        answer, log = react(question, guard=guard)
        steps += len(log)                                    # model and script steps
        parsed += sum(why in ("ok", "repeat") for _, why in log)   # steps whose action parsed
        finished += answer is not None
        ok = answer is not None and check(answer)
        correct += ok
        print(f"{'right' if ok else 'WRONG':5s}  {len(log)} steps  {question:50s} -> {str(answer)[:70]}")
    print(f"\nparsed {parsed}/{steps} steps, finished {finished}/{len(TASKS)} tasks, right {correct}/{len(TASKS)}")
    return parsed, steps, finished, correct

t0 = time.time()
results_plain = evaluate()                                   # no guard
print(f"({time.time() - t0:.0f} s)")

**What to look for.** When we ran this on a CPU, all 18 of the model's actions parsed, it finished 7 of the 8 tasks, and 7 of the 8
answers passed our check. Seven of the tasks take two steps each: fetch, then finish. The eighth is the loop from section 2.4.
So the format is not the difficulty at this size, and the lesson moves to where the clip puts it. Read the answers rather than the
score. The creatinine answer reports "1.1, 1.3, 1.6 mmol/L" when creatinine is in mg/dL, and our check passes it, exactly as
the note above warned it would: the agent was right about the trend and wrong about the unit, and nothing in the loop noticed.
An agent that is right seven times out of eight is not an agent you can leave unwatched; it is one whose failures are now rare
enough to be easy to miss. Your numbers will differ with another model or another library version.

## 3. Errors compound over steps  *(clip 3)*

### 3.1 Simulating 0.95ⁿ and 0.99ⁿ
If each step succeeds with probability p and steps fail independently, a task of n steps succeeds with pⁿ. We check the formula
with a simulation of 100,000 tasks for each case. Independence is an assumption made for teaching; real failures are often
correlated.

In [ ]:
for p in (0.95, 0.99):
    for n in (5, 10, 20):
        sim = (rng.random((100_000, n)) < p).all(axis=1).mean()   # a task succeeds only if every step does
        print(f"p = {p}, {n:2d} steps: formula {p ** n:.2f}, simulation {sim:.2f}")

ns = np.arange(1, 31)                                        # 1 to 30 steps
for p, c in ((0.95, "#9CA3AF"), (0.99, "#0021A5")):
    plt.plot(ns, p ** ns, "o-", color=c, ms=3, label=f"p = {p} per step")
plt.xlabel("steps in the task"); plt.ylabel("P(task succeeds)"); plt.ylim(0, 1.02)
plt.legend(); plt.grid(alpha=0.3); plt.show()

### 3.1b Watch a run of twenty steps

Section 3.1 printed the curve. The animation runs the agent instead, which makes the same point harder to argue with.

The top row is one trajectory: each box is a step, green if it went right and orange if it did not, sampled at 95%
per-step reliability. The run is counted as a success only if the whole row is green, because an agent that books the
wrong appointment at step 11 has failed no matter how good steps 1 to 10 were.

The bottom panel keeps score over many runs and draws the two curves beside the observed rate. At twenty steps, 95% per
step is a coin flip for the trajectory, and the gap to the 99% curve is the entire argument for fewer steps, checked
results, and a clinician at the end of the chain.

In [ ]:
# An animation, drawn by the code below. It renders with to_jshtml, which needs no
# ffmpeg and plays in Colab, in Jupyter, and on nbviewer. Press play, or drag the
# slider to step through it frame by frame.
import numpy as np                                           # arrays
import matplotlib.pyplot as plt                              # figures
from matplotlib.animation import FuncAnimation               # the animation itself
from IPython.display import HTML                             # shows it in the notebook

UF, ORANGE, GREY = "#0021A5", "#FA4616", "#6B7280"           # the course colours
plt.rcParams["figure.dpi"] = 78                              # keeps the rendered animation light

STEPS, RUNS, P95, P99 = 20, 400, 0.95, 0.99                  # steps per run, runs, the two reliabilities
rng = np.random.default_rng(7)
trajectories = rng.random((RUNS, STEPS)) < P95               # True where a step went right
first_fail = np.where(trajectories.all(1), STEPS, trajectories.argmin(1))   # where each run first broke
ns = np.arange(1, STEPS + 1)

fig, (row, curve) = plt.subplots(2, 1, figsize=(9.0, 4.2),
                                 gridspec_kw={"height_ratios": [0.7, 1.3]})
fig.subplots_adjust(hspace=0.55)

def draw(f):
    r = f                                                    # the run being shown
    row.clear(); curve.clear()

    # ---- top: one trajectory, step by step
    ok = trajectories[r]
    row.set_xlim(-0.5, STEPS - 0.5); row.set_ylim(-0.6, 0.6); row.axis("off")
    for s in range(STEPS):
        broke = s >= first_fail[r]                           # everything from the first failure is wasted
        row.add_patch(plt.Rectangle((s - 0.42, -0.3), 0.84, 0.6,
                                    fc=("#E5E7EB" if broke else "#1F9D55") if s != first_fail[r] else ORANGE,
                                    ec="white", lw=1.2))
        row.text(s, 0, str(s + 1), ha="center", va="center", fontsize=8,
                 color="white" if not broke or s == first_fail[r] else "#9CA3AF")
    done = ok.all()
    row.set_title(f"run {r + 1}: " + ("all {} steps correct".format(STEPS) if done
                                      else f"first wrong step at {first_fail[r] + 1}"),
                  loc="left", fontsize=11, color="#1F9D55" if done else ORANGE)

    # ---- bottom: the observed success rate against the two curves
    seen = trajectories[:r + 1]
    obs = [(seen[:, :k].all(1)).mean() for k in ns]          # fraction of runs still correct after k steps
    curve.plot(ns, P95 ** ns, color=ORANGE, lw=2.0, label="0.95 per step")
    curve.plot(ns, P99 ** ns, color=UF, lw=2.0, label="0.99 per step")
    curve.plot(ns, obs, color="#111827", lw=1.4, ls=":", label=f"observed, {r + 1} runs")
    curve.set_xlim(1, STEPS); curve.set_ylim(0, 1.02)
    curve.set_xlabel("steps in the trajectory", fontsize=10)
    curve.set_ylabel("whole run correct", fontsize=10)
    curve.legend(loc="lower left", fontsize=9, frameon=False)
    curve.spines[["top", "right"]].set_visible(False)
    curve.set_title(f"at {STEPS} steps: 0.95 gives {P95 ** STEPS:.2f}, 0.99 gives {P99 ** STEPS:.2f}",
                    loc="right", fontsize=10, color=GREY)
    return []

ani = FuncAnimation(fig, draw, frames=48, interval=140)
plt.close(fig)
HTML(ani.to_jshtml(fps=7))

### 3.2 Mitigation: check results and stop repeated calls
The loop in section 2.5 looks easy to catch in the program: if the agent repeats a call with exactly the same arguments, the
program does not run the tool again and tells the agent to use the result it has. We rerun all eight tasks with this guard.

In [ ]:
t0 = time.time()
results_guard = evaluate(guard=True)                         # the same eight tasks with the repeat guard
print(f"({time.time() - t0:.0f} s)")

When we ran it, the guard changed nothing at all: the same 18 of 18 actions parsed, the same 7 of 8 tasks finished, the same 7 of
8 answers were right, and the potassium task still ran to the step limit. The guard did fire. From the second step on, the
program stopped re-running `get_labs` and replaced the observation with a message telling the agent to use the result it already
had. The model repeated the call anyway, three more times, because it was not reading the observation in the first place; it was
continuing from the `Observation:` line it had written itself. That is the honest shape of this kind of mitigation. A guard can
stop the program from acting on a repeated call, which is worth having, but it cannot make the model read. For that you need a
check of the answer against the data (for example a rule comparing each value with its reference range, which would also have
caught the creatinine unit) or a stronger model, and at some point a person.

### 3.3 Mitigation: read-only tools and a clinician checkpoint
The program only runs tools in `TOOLS`, and both of them read. A call to a tool that writes is rejected by the parser before it
can do anything. If a tool that changes care were ever added, it would go through a checkpoint where a clinician approves or
rejects it; here the approval function is a stand-in that always says no.

In [ ]:
print(parse_action('Action: {"tool": "place_order", "args": {"patient_id": "P-001", "order": "repeat potassium and ECG"}}'))

def clinician_approves(action):
    """Stand-in for a person reviewing a proposed action; in this notebook nothing is ever approved."""
    print("proposed for review:", action)
    return False

proposal = {"tool": "place_order", "args": {"patient_id": "P-001", "order": "repeat potassium and ECG"}}   # the clip's example
print("placed" if clinician_approves(proposal) else "not placed: waiting for a clinician")
print(f"\nTotal run time: {time.time() - T_START:.0f} s")

## Recap

| Clip | Code in this notebook |
|---|---|
| 1. Retrieval-augmented generation | `embed`, `retrieve` (cosine scores), a threshold `LAMBDA` chosen for this model, `rag_answer` with fallback or abstain |
| 2. Reason, act, observe | `get_labs` and `get_meds`, `parse_action`, the few-shot `SYSTEM` prompt, `react`, and `evaluate`: parsed, finished, right |
| 3. Errors compound over steps | A simulation of pⁿ, the repeat guard, read-only tools, and `clinician_approves` |

> **Try it (optional):** ask P-001 whether a medication could explain the potassium (it takes lisinopril and spironolactone,
> which both raise potassium): the agent needs two tools. Remove the worked example from `SYSTEM` and rerun section 2.5. Add a
> rule that checks each answer against `LIMITS` and see how many wrong answers it catches.